# Final Project

ST554: Analysis of Big Data Spring 2026

Ryan Mersereau

## Goals

For this project, we'll use spark to handle streaming data and fit a machine learning model, among other things.

The data is modified from the UCI machine learning repository. The study was about relating power
consumption from different zones of Tetouan city to various factors like time of day, temperature, and
humidity.
* We’ll have a chunk of the (modified) data to build a model on.
* We will then ‘stream data’ to a folder in the form of CSV files. We'll be monitoring this folder. When
data comes in and use a fitted model to predict on the incoming data!

## Fitting the Model

The file `power_ml_data.csv`comes from [here](https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv), we will read this data into a standard pandas dataframe and convert this into a spark data frame.

First, let's import needed modules and start our Spark Session

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Binarizer, OneHotEncoder, StringIndexer,
    VectorAssembler, PCA, SQLTransformer
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("PowerZoneFinal").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 14:08:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 14:08:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/28 14:08:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
# Read data into pandas, convert to spark df
url = "https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv"
pdf = pd.read_csv(url)

df = spark.createDataFrame(pdf)
df.printSchema()
df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We can see our dataframe has 10 columns. Here we'll be using `Power_Zone_3` as our response variable, and all other variables as predictors (in case `Power_Zone_3` were to go offline).

We want to fit an elastic net model using CV, and will start by performing some transformations using `MLlib` functions that can be put in a pipeline.

### Transformations

To start, we need to cast our`Hour`variable as`DoubleType`(currently`long`) and then binarize this column into night vs day.

In [3]:
sql_transformer = SQLTransformer(
    statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__"
)

# binarize hour (threshold = 6.5, night = 0, day = 1)
binarizer = Binarizer(
    threshold=6.5,
    inputCol="Hour_double",
    outputCol="Hour_bin"
)

Next, we need to One-hot encode the`Month`column. This also converts these month categories into binary format by creating separate column for each category.

In [4]:
ohe = OneHotEncoder(
    inputCols=["Month"],
    outputCols=["Month_ohe"],
    dropLast=True   
)

Now lets run a PCA (Principal component analysis) fit on the `Temperature`,`Humidity`,`Wind_Speed`,`General_Diffuse_Flows`, and`Diffuse_Flows`columns. 

We'll do this by placing these variables in a column together with a `VectorAssembler()`and then use this with the `PCA()`estimator. We'll use two PCs in our transformation.

In [5]:
weather_cols = ["Temperature", "Humidity", "Wind_Speed",
                "General_Diffuse_Flows", "Diffuse_Flows"]

weather_assembler = VectorAssembler(
    inputCols=weather_cols,
    outputCol="weather_features"
)

# Fit PCA with 2 components
pca = PCA(k=2, inputCol="weather_features", outputCol="pca_features")

Now, we can rename the response variable (`Power_Zone_3`) as`label` and use`VectorAssembler()`to put our predictors as`features`.

We will use the:
* two fitted PCA features
* binary `Hour` variable
* `Power_Zone_1`
* `Power_Zone_2`
* `Month` indicator variables

In [6]:
label_transformer = SQLTransformer(
    statement="SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

In [7]:
# VectorAssembler
feature_assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin",
               "Power_Zone_1", "Power_Zone_2", "Month_ohe"],
    outputCol="features"
)

### Pipeline

Now, we will combine all our prior transformations together to build a pipeline.

In [8]:
pipeline = Pipeline(stages=[
    sql_transformer,    
    binarizer,          
    ohe,                
    weather_assembler,  
    pca,                
    label_transformer,  
    feature_assembler   
])